# per-rank-cuda-device — worked example 3: Verify no-shared-device invariant across all ranks

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `per-rank-cuda-device`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A fundamental invariant in multi-GPU training is that no two ranks share the same CUDA device index. If two ranks were pinned to `cuda:0`, they would overwrite each other's gradients and produce incorrect parameter updates. Building all per-rank contexts and asserting that the set of device indices has size equal to `world_size` programmatically verifies this invariant.

## Worked solution

**Step 1 — Build contexts for all ranks.** Call `build_rank_context(r, world_size)` for each `r` in `range(world_size)` to get a list of context dicts.

**Step 2 — Extract device indices.** `{ctx['device'].index for ctx in ctxs}` builds a set of integer GPU indices. If any two ranks share a device, the set will be smaller than `world_size`.

**Step 3 — Assert uniqueness.** `assert len(device_indices) == world_size` catches any collision. We also assert the indices are exactly `{0, 1, ..., world_size-1}` — no gaps, no repeats.

**Step 4 — Additional sanity checks.** Exactly one rank has `is_master == True` (rank 0), and each rank's `device_str` is `'cuda:{rank}'`.

In [ ]:
import torch as t

def build_rank_context(rank: int, world_size: int) -> dict:
    device = t.device(f'cuda:{rank}')
    return {
        'rank': rank,
        'world_size': world_size,
        'device': device,
        'is_master': rank == 0,
        'device_str': f'cuda:{rank}',
    }

def verify_no_shared_devices(world_size: int) -> bool:
    """Build all rank contexts and verify the no-shared-device invariant."""
    ctxs = [build_rank_context(r, world_size) for r in range(world_size)]
    device_indices = {ctx['device'].index for ctx in ctxs}
    # Uniqueness check
    all_unique = len(device_indices) == world_size
    # Expected indices are 0, 1, ..., world_size-1
    expected = set(range(world_size))
    correct_indices = device_indices == expected
    # Exactly one master
    masters = [ctx for ctx in ctxs if ctx['is_master']]
    one_master = len(masters) == 1 and masters[0]['rank'] == 0
    return all_unique and correct_indices and one_master

for ws in [1, 2, 4, 8]:
    ok = verify_no_shared_devices(ws)
    print(f'world_size={ws}: invariant holds = {ok}')  # True for all